# Credit Risk Classifier — EDA & Model Performance

**Goal:** Predict loan applicant credit risk tier (P1–P4)

**Pipeline:** Preprocessing → Feature Selection → Encoding → Model Training

---
## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import sys, os
os.makedirs("../results", exist_ok=True)
sys.path.append("../src")
from data_preprocessing import run_preprocessing
from feature_selection   import run_feature_selection
from encoding            import run_encoding
from model_training      import run_training

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
COLORS = ["#4C72B0","#DD8452","#55A868","#C44E52"]
print("Setup complete ✅")

---
## 1. Load Data

In [ ]:
df_raw = run_preprocessing()
print(f"Shape: {df_raw.shape}")
df_raw.head()

---
## 2. EDA
### 2.1 Target Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
counts = df_raw["Approved_Flag"].value_counts().reindex(["P1","P2","P3","P4"])
axes[0].bar(counts.index, counts.values, color=COLORS, edgecolor="white")
axes[0].set_title("Applicant Count per Risk Tier", fontweight="bold")
axes[0].set_xlabel("Risk Tier"); axes[0].set_ylabel("Count")
for bar, val in zip(axes[0].patches, counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+50, f"{val:,}", ha="center", fontsize=10)
axes[1].pie(counts.values, labels=counts.index, autopct="%1.1f%%", colors=COLORS,
            startangle=140, wedgeprops={"edgecolor":"white","linewidth":1.5})
axes[1].set_title("Class Distribution (%)", fontweight="bold")
plt.tight_layout()
plt.savefig("../results/target_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

### 2.2 Categorical Features vs Target

In [ ]:
cat_features = ["MARITALSTATUS", "EDUCATION", "GENDER"]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col in zip(axes, cat_features):
    ct = pd.crosstab(df_raw[col], df_raw["Approved_Flag"], normalize="index")[["P1","P2","P3","P4"]]
    ct.plot(kind="bar", ax=ax, color=COLORS, edgecolor="white", width=0.75)
    ax.set_title(f"{col} vs Risk Tier", fontweight="bold")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
    ax.tick_params(axis="x", rotation=30)
    ax.legend(title="Risk Tier", fontsize=8)
plt.tight_layout()
plt.savefig("../results/categorical_eda.png", dpi=150, bbox_inches="tight")
plt.show()

### 2.3 Correlation Heatmap

In [ ]:
num_df = df_raw[[c for c in df_raw.columns if df_raw[c].dtype != "object" and c not in ["PROSPECTID","Approved_Flag"]]]
corr = num_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
plt.figure(figsize=(14, 10))
sns.heatmap(corr, mask=mask, cmap="coolwarm", center=0, linewidths=0.3, cbar_kws={"shrink":0.7})
plt.title("Feature Correlation Heatmap", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../results/correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 3. Run Full Pipeline

In [ ]:
df_sel, num_cols = run_feature_selection(df_raw)
df_enc = run_encoding(df_sel)
xgb_model, label_encoder = run_training(df_enc)
print("Pipeline done ✅")

---
## 4. Confusion Matrix — XGBoost

In [ ]:
from sklearn.metrics import confusion_matrix
y = df_enc["Approved_Flag"]; X = df_enc.drop(["Approved_Flag"], axis=1)
y_enc = label_encoder.transform(y)
X_train, X_test, y_train, y_test = train_test_split(X, y_enc, test_size=0.2, random_state=42)
y_pred = xgb_model.predict(X_test)
class_names = label_encoder.classes_

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
import seaborn as sns
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_title("Confusion Matrix (Counts)", fontweight="bold")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
cm_norm = confusion_matrix(y_test, y_pred, normalize="true")
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", xticklabels=class_names, yticklabels=class_names, vmin=0, vmax=1, ax=axes[1])
axes[1].set_title("Confusion Matrix (Normalised)", fontweight="bold")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Actual")
plt.suptitle("XGBoost — Confusion Matrices", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("../results/confusion_matrix_xgb.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 5. Model Comparison — Precision / Recall / F1

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
import xgboost as xgb
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

le = LabelEncoder(); y_enc2 = le.fit_transform(df_enc["Approved_Flag"])
X2 = df_enc.drop(["Approved_Flag"], axis=1)
X_tr, X_te, y_tr, y_te = train_test_split(X2, y_enc2, test_size=0.2, random_state=42)

models = {
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "XGBoost"      : xgb.XGBClassifier(objective="multi:softmax", num_class=4, eval_metric="mlogloss"),
    "Decision Tree": DecisionTreeClassifier(max_depth=20, min_samples_split=10)
}
results = {}
for name, m in models.items():
    m.fit(X_tr, y_tr); yp = m.predict(X_te)
    acc = accuracy_score(y_te, yp); p,r,f,_ = precision_recall_fscore_support(y_te, yp)
    results[name] = {"accuracy":acc, "precision":p, "recall":r, "f1":f}
    print(f"{name:20s} → Accuracy: {acc:.4f}")

In [ ]:
classes = le.classes_; model_names = list(results.keys())
x = np.arange(len(classes)); width = 0.25
model_colors = ["#4C72B0","#DD8452","#55A868"]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric, title in zip(axes, ["precision","recall","f1"], ["Precision","Recall","F1-Score"]):
    for idx, (name, color) in enumerate(zip(model_names, model_colors)):
        ax.bar(x+idx*width, results[name][metric], width, label=name, color=color, alpha=0.85, edgecolor="white")
    ax.set_xticks(x+width); ax.set_xticklabels(classes); ax.set_ylim(0,1.1)
    ax.set_title(f"{title} per Class", fontweight="bold"); ax.legend(fontsize=8)
    ax.axhline(0.8, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
plt.suptitle("Model Comparison: Precision · Recall · F1", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("../results/model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 6. Feature Importance — XGBoost

In [ ]:
importances = models["XGBoost"].feature_importances_
feat_df = pd.DataFrame({"Feature":X2.columns,"Importance":importances}).sort_values("Importance",ascending=False).head(15)
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(feat_df["Feature"][::-1], feat_df["Importance"][::-1], color="#4C72B0", edgecolor="white")
ax.set_xlabel("Importance Score")
ax.set_title("XGBoost — Top 15 Feature Importances", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../results/feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()